In [1]:
!pip install flask flask_cors pyngrok

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import tensorflow as tf
model = tf.keras.models.load_model('/content/drive/MyDrive/stroke.h5')

In [5]:
!unzip templates.zip

Archive:  templates.zip
   creating: templates/
  inflating: templates/index.html    
  inflating: templates/results.html  


In [6]:
import os
from pyngrok import ngrok
os.environ["NGROK_AUTH_TOKEN"] = "2uVUEedTUZ9DQBNhLq3COUp51mz_6MidPkUHxm4A8v3WyS1Jg"
ngrok.set_auth_token("2uVUEedTUZ9DQBNhLq3COUp51mz_6MidPkUHxm4A8v3WyS1Jg")

In [8]:
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
import tensorflow as tf
import numpy as np
import cv2
import os
import threading
from pyngrok import ngrok

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

@app.after_request
def add_cors_headers(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type'
    return response

def preprocess_image(img):
    img = cv2.imread(str(img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255
    img = cv2.resize(img, (224, 224))
    return img

@app.route('/')
def index():
    return render_template('index.html')

@app.route("/analyze", methods=['POST'])
def analyze_image():
    try:
        if 'image' not in request.files:
            return "لم يتم إرسال أي صورة", 400

        image_file = request.files['image']
        save_path = os.path.join(os.getcwd(), 'uploaded_images', "test.png")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        image_file.save(save_path)

        processed_image = preprocess_image(save_path)
        processed_image = np.expand_dims(processed_image, axis=0)

        prediction = model.predict(processed_image)[0][0]

        # تحديد التصنيف والثقة
        has_stroke = prediction >= 0.5
        confidence = prediction * 100 if has_stroke else (1 - prediction) * 100

        # إرجاع النتيجة إلى صفحة results.html
        return render_template(
            'results.html',
            has_stroke=has_stroke,
            confidence=confidence
        )

    except Exception as e:
        return f"حدث خطأ أثناء تحليل الصورة: {str(e)}", 500
def run_flask(port):
    """تشغيل التطبيق على منفذ محدد"""
    app.run(host='0.0.0.0', port=port, use_reloader=False)

if __name__ == "__main__":

    port = int("8" + str(np.random.randint(9)) + str(np.random.randint(9)) + str(np.random.randint(9)))

    # تشغيل Flask في خيط منفصل
    thread = threading.Thread(target=lambda: run_flask(port))
    thread.daemon = True
    thread.start()

    # إنشاء نفق ngrok
    public_url = ngrok.connect(port)
    print(f'يمكنك الوصول للتطبيق من خلال الرابط التالي: {public_url}')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8065
 * Running on http://172.28.0.12:8065
INFO:werkzeug:Press CTRL+C to quit


يمكنك الوصول للتطبيق من خلال الرابط التالي: NgrokTunnel: "https://be62-34-147-9-28.ngrok-free.app" -> "http://localhost:8065"
